# 12.1 Motivation
The baseline CNN showed moderate performance, particularly confusion between disease classes. To improve feature extraction, a deep pretrained network (ResNet50) is introduced using transfer learning. ResNet architectures are effective at learning hierarchical visual features and often outperform shallow CNNs on small datasets.

In [ ]:
# 🌿 Swiss chard Leaf Disease Classification using resnet-50

## 1. Imports & Configuration
## Import libaries
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
from sklearn.preprocessing import label_binarize
from tensorflow.keras.preprocessing.image import load_img, img_to_array

In [ ]:
## 2. Load Dataset

images = []
labels = []

class_names = sorted(os.listdir(DATA_DIR))
class_to_index = {cls: idx for idx, cls in enumerate(class_names)}

for cls in class_names:
    cls_path = os.path.join(DATA_DIR, cls)
    for img_name in os.listdir(cls_path):
        img_path = os.path.join(cls_path, img_name)
        img = load_img(img_path, target_size=IMAGE_SIZE)
        img = img_to_array(img)
        images.append(img)
        labels.append(class_to_index[cls])

images = np.array(images)
labels = np.array(labels)

print("Classes:", class_names)
print("Total images:", len(images))

## 3. Train / Validation / Test Split

X_temp, X_test, y_temp, y_test = train_test_split(
    images, labels, test_size=0.2, random_state=RANDOM_STATE, stratify=labels
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.2, random_state=RANDOM_STATE, stratify=y_temp
)

print("Train:", len(X_train))
print("Validation:", len(X_val))
print("Test:", len(X_test))

In [ ]:

## 4. TensorFlow Datasets


train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
val_ds   = tf.data.Dataset.from_tensor_slices((X_val, y_val))
test_ds  = tf.data.Dataset.from_tensor_slices((X_test, y_test))

train_ds = train_ds.shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds   = val_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds  = test_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


In [ ]:
## 5. Data Augmentation

data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
])


In [ ]:
### 12.2 ResNet50 Model Definition

from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input

# Data pipeline with ResNet preprocessing
train_ds_resnet = train_ds.map(lambda x, y: (preprocess_input(x), y))
val_ds_resnet   = val_ds.map(lambda x, y: (preprocess_input(x), y))
test_ds_resnet  = test_ds.map(lambda x, y: (preprocess_input(x), y))

base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=IMAGE_SIZE + (3,)
)

base_model.trainable = False  # Freeze pretrained layers

resnet_model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

resnet_model.summary()

In [ ]:
### 12.3 Compile and Train ResNet Model


resnet_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

resnet_early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

resnet_history = resnet_model.fit(
    train_ds_resnet,
    validation_data=val_ds_resnet,
    epochs=EPOCHS,
    callbacks=[resnet_early_stopping]
)



In [ ]:
### 12.4 Training vs Validation Curves (ResNet50)

plt.figure(figsize=(6,4))
plt.plot(resnet_history.history['accuracy'], label='Training Accuracy')
plt.plot(resnet_history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.title("ResNet50 Training and Validation Accuracy")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(6,4))
plt.plot(resnet_history.history['loss'], label='Training Loss')
plt.plot(resnet_history.history['val_loss'], label='Validation Loss')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("ResNet50 Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
### 12.5 Test Evaluation (ResNet50)

resnet_test_loss, resnet_test_accuracy = resnet_model.evaluate(test_ds_resnet)
print(f"ResNet50 Test Accuracy: {resnet_test_accuracy:.4f}")
print(f"ResNet50 Test Loss: {resnet_test_loss:.4f}")

In [ ]:
### 12.6 Confusion Matrix (ResNet50)

y_true_resnet, y_pred_resnet = [], []

for x, y in test_ds_resnet:
    preds = resnet_model.predict(x)
    y_pred_resnet.extend(np.argmax(preds, axis=1))
    y_true_resnet.extend(y.numpy())

cm_resnet = confusion_matrix(y_true_resnet, y_pred_resnet)

plt.figure(figsize=(6,5))
sns.heatmap(cm_resnet, annot=True, fmt='d', cmap='Greens',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("ResNet50 Confusion Matrix")
plt.show()

print(classification_report(y_true_resnet, y_pred_resnet, target_names=class_names))

In [ ]:

### 12.7 ROC Curves (ResNet50)
y_true_bin_resnet = label_binarize(y_true_resnet, classes=[0,1,2])
y_score_resnet = resnet_model.predict(test_ds_resnet)

plt.figure(figsize=(7,6))
for i, cls in enumerate(class_names):
    fpr, tpr, _ = roc_curve(y_true_bin_resnet[:, i], y_score_resnet[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{cls} (AUC = {roc_auc:.2f})")

plt.plot([0,1], [0,1], 'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ResNet50 ROC Curves")
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

In [ ]:
## 14. Save ResNet Model
resnet_model.save("swiss_cahrd_leaf_disease_cnn_vs_resnet50.keras")